# FrameWeave Ovis2.5-9B — multi-seed HERBench lite_v2

Notebook **chỉ eval**. Không train. Oracle $K=8$, greedy, parser chữ cái. Output sai vẫn ở mẫu số.

Ba biến thể `global_only`, `local_only`, `hybrid` trên seed **1, 100, 2027**. Cùng manifest 1.971 câu / 68 video. Run All eval từng seed rồi ghi bảng từng seed và mean ± std.

## Source code

Mở notebook này từ GitHub hoặc Colab. Run All clone [FrameWeave](https://github.com/nqMinh-GR/FrameWeave.git) vào `/content/FrameWeave`, rồi symlink `ovis/reljoin`, `ablation/`, `configs/` và `run_ablation.py` vào `/content`. Không upload file source.

Checkpoint lấy từ Drive. Đường dẫn đủ là `/content/drive/MyDrive/frameweave_ovis_multiseed_outputs/seed_<seed>/seed_<seed>/<variant>/best/adapter.pt`. Không tải PerceptionTest.

## Hardware

Colab **A100 40 GB**. Disk trống ≥ ~55 GB (Ovis2.5-9B ~18 GB + HERBench videos ~35 GB).

Eval 1.971 câu × 9 checkpoint là một session riêng. Prediction đã có được nối tiếp khi Run All chạy lại.


## 1. Configuration


In [ ]:
# 1. Eval-only config — seeds 1, 100, 2027
from __future__ import annotations
from pathlib import Path

assert Path("/content").exists(), "Notebook này chỉ chạy trên Google Colab (/content)."

ROOT = Path("/content")
SEEDS = [1, 100, 2027]
VARIANT = "global_only,local_only,hybrid"
VARIANT_LIST = ["global_only", "local_only", "hybrid"]

MODEL_ID = "ATH-MaaS/Ovis2.5-9B"
MODEL_REVISION = "5a1f9b995c98ae5b85a28ec89d88d74295be0ea7"
HERBENCH_ID = "DanBenAmi/HERBench"
HERBENCH_REVISION = "30b42e97180f2c7c0fdc3f2aed412ae04f0366ca"
HERBENCH_BASE_SHA256 = "6104884c366906d5c223cc9727c3bad778dcdf98c881c4b17b556c20e940af97"
HERBENCH_ACC_SHA256 = "eb67b618f3004b52c8290b2fd30e4012153a66f3f93d576c29760280caa3344d"
HERBENCH_BASE_URL = "https://raw.githubusercontent.com/DanBenAmi/HERBench/main/evaluation/model_wrappers/base_vlm.py"
HERBENCH_ACC_URL = "https://raw.githubusercontent.com/DanBenAmi/HERBench/main/evaluation/calculate_accuracy.py"

VENV = ROOT / "ovis_env"
PYTHON = VENV / "bin" / "python"
OUT_ROOT = ROOT / "outputs" / "frameweave_ovis_multiseed"
MODEL_DIR = ROOT / "checkpoints" / "ovis2_5_9b"
HERBENCH_ROOT = ROOT / "HERBench_official"
HERBENCH_VIDEOS = HERBENCH_ROOT / "videos"
EVAL_DIR = ROOT / "herbench_eval_pins"
PERSIST = Path("/content/drive/MyDrive/frameweave_ovis_multiseed_outputs")
CONFIG_DIR = ROOT / "configs" / "multiseed"

print({
    "SEEDS": SEEDS,
    "VARIANT": VARIANT,
    "OUT_ROOT": str(OUT_ROOT),
    "MODEL_DIR": str(MODEL_DIR),
    "backbone": MODEL_ID,
})


## 2. Environment (`ovis_env`)

Official Ovis2.5-9B pins: `torch==2.4.0`, `transformers==4.51.3`, `flash-attn==2.7.0.post2`.
Stop if flash-attn install fails — the ViT hard-imports it.


In [ ]:
# 2. Bootstrap isolated ovis_env (Python 3.11 + torch 2.4.0 + flash-attn 2.7)
from __future__ import annotations
import os, subprocess
from pathlib import Path

assert Path("/content").exists(), "This notebook targets Google Colab (/content)."
ROOT = Path("/content")
VENV = ROOT / "ovis_env"
VENV_BIN = VENV / "bin"
PYTHON = VENV_BIN / "python"
PIP = VENV_BIN / "pip"
LOG = ROOT / "outputs" / "frameweave_ovis_multiseed" / "logs" / "cell4_env.log"
LOG.parent.mkdir(parents=True, exist_ok=True)
LOG.write_text("", encoding="utf-8")
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def sh(cmd, check=True, env=None):
    line = "$ " + " ".join(map(str, cmd))
    print(line, flush=True)
    with LOG.open("a", encoding="utf-8") as f:
        f.write(line + "\n")
        proc = subprocess.run(
            list(map(str, cmd)),
            check=False,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            env=env,
        )
        f.write(proc.stdout or "")
        f.write(f"\nexit={proc.returncode}\n\n")
    print(proc.stdout or "", end="")
    if check and proc.returncode != 0:
        print("LOG:", LOG)
        print((proc.stdout or "")[-4000:])
        raise RuntimeError(f"command failed ({proc.returncode}). See {LOG}")
    return proc

sh(["apt-get", "update", "-qq"], check=False)
sh(["apt-get", "install", "-y", "-qq", "python3.11", "python3.11-venv", "python3.11-dev", "ffmpeg"], check=False)
py311 = Path("/usr/bin/python3.11")
assert py311.exists(), "Không tìm thấy python3.11 — kiểm tra apt install."

need_recreate = True
if PYTHON.exists():
    ver = subprocess.check_output([str(PYTHON), "-V"], text=True).strip()
    print("Existing venv:", ver)
    need_recreate = "3.11" not in ver
if need_recreate:
    if VENV.exists():
        sh(["rm", "-rf", str(VENV)])
    sh([str(py311), "-m", "venv", "--copies", str(VENV)])
if not PIP.exists():
    sh([str(PYTHON), "-m", "ensurepip", "--upgrade"], check=False)
if not PIP.exists():
    get_pip = Path("/tmp/get-pip.py")
    sh(["curl", "-sSL", "https://bootstrap.pypa.io/get-pip.py", "-o", str(get_pip)])
    sh([str(PYTHON), str(get_pip)])
sh([str(PYTHON), "-m", "pip", "install", "-q", "--upgrade", "pip", "setuptools<70", "wheel"])
print("Python in venv:", subprocess.check_output([str(PYTHON), "-V"], text=True).strip())
sh([
    str(PYTHON), "-m", "pip", "install", "--disable-pip-version-check",
    "torch==2.4.0", "torchvision==0.19.0",
    "--index-url", "https://download.pytorch.org/whl/cu121",
])
sh([
    str(PYTHON), "-m", "pip", "install", "--disable-pip-version-check", "--upgrade-strategy", "only-if-needed",
    "transformers==4.51.3", "numpy==1.25.0", "pillow==10.3.0", "moviepy==1.0.3",
    "huggingface_hub", "hf-transfer", "datasets", "pyarrow==25.0.1", "accelerate", "einops", "sentencepiece",
    "opencv-python-headless", "pandas", "scipy", "tqdm", "ninja", "packaging",
])
print("Installing flash-attn==2.7.0.post2 (required by Ovis ViT)...")
fa_env = os.environ.copy()
fa_env["MAX_JOBS"] = "4"
sh(
    [str(PYTHON), "-m", "pip", "install", "flash-attn==2.7.0.post2", "--no-build-isolation"],
    env=fa_env,
)
check = subprocess.check_output([
    str(PYTHON), "-c",
    "import torch, transformers, flash_attn, pyarrow, numpy; "
    "print(torch.__version__); print(transformers.__version__); print(flash_attn.__version__); "
    "print('numpy', numpy.__version__); print('pyarrow', pyarrow.__version__); "
    "assert numpy.__version__.startswith('1.'), numpy.__version__; "
    "assert pyarrow.__version__ == '25.0.1', pyarrow.__version__; "
    "print(torch.__file__); print(transformers.__file__)"
], text=True)
print(check)
for line in check.splitlines():
    if "/site-packages/" in line:
        assert "/content/ovis_env/" in line, f"Package leaked outside venv: {line}"
print("ovis_env ready:", PYTHON)
print("LOG:", LOG)


## 3. Drive + pin HERBench evaluators

Gắn Drive và pin evaluator. Checkpoint của từng seed được nối từ repo sau khi clone; Drive chỉ bù file còn thiếu.


In [ ]:
# 3. Mount Drive and pin the HERBench evaluator. Seed checkpoints are linked after the clone.
from __future__ import annotations

import hashlib
import os
import time
import urllib.request
from pathlib import Path

ROOT = Path("/content")
HERBENCH_ROOT = ROOT / "HERBench_official"
HERBENCH_VIDEOS = HERBENCH_ROOT / "videos"
MODEL_DIR = ROOT / "checkpoints" / "ovis2_5_9b"
OUT_ROOT = ROOT / "outputs" / "frameweave_ovis_multiseed"
EVAL_DIR = ROOT / "herbench_eval_pins"
HERBENCH_BASE_URL = "https://raw.githubusercontent.com/DanBenAmi/HERBench/main/evaluation/model_wrappers/base_vlm.py"
HERBENCH_ACC_URL = "https://raw.githubusercontent.com/DanBenAmi/HERBench/main/evaluation/calculate_accuracy.py"
HERBENCH_BASE_SHA256 = "6104884c366906d5c223cc9727c3bad778dcdf98c881c4b17b556c20e940af97"
HERBENCH_ACC_SHA256 = "eb67b618f3004b52c8290b2fd30e4012153a66f3f93d576c29760280caa3344d"
JSDELIVR_BASE = "https://cdn.jsdelivr.net/gh/DanBenAmi/HERBench@main/evaluation/model_wrappers/base_vlm.py"
JSDELIVR_ACC = "https://cdn.jsdelivr.net/gh/DanBenAmi/HERBench@main/evaluation/calculate_accuracy.py"

try:
    SEEDS
except NameError:
    SEEDS = [1, 100, 2027]

for path in [HERBENCH_ROOT, HERBENCH_VIDEOS, MODEL_DIR.parent, EVAL_DIR, OUT_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PERSIST = Path("/content/drive/MyDrive/frameweave_ovis_multiseed_outputs")
except Exception as exc:
    print("Drive mount skipped:", exc)
    PERSIST = OUT_ROOT / "persist_local"
PERSIST.mkdir(parents=True, exist_ok=True)
os.environ["RELJOIN_PERSIST"] = str(PERSIST)
os.environ["RELJOIN_ROOT"] = str(ROOT)
os.environ["OVIS_ABLATION_ROOT"] = str(ROOT)
os.environ["PYTHONUNBUFFERED"] = "1"


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def fetch_pinned(urls: list[str], dest: Path, expected: str) -> None:
    if dest.exists() and sha256_file(dest) == expected:
        print("Already pinned:", dest.name)
        return
    last_err = None
    for url in urls:
        for attempt in range(4):
            try:
                print(f"GET {url} (try {attempt + 1})")
                urllib.request.urlretrieve(url, dest)
                got = sha256_file(dest)
                if got == expected:
                    print("OK", dest.name)
                    return
                last_err = RuntimeError(f"SHA mismatch for {dest.name}: {got}")
            except Exception as exc:
                last_err = exc
                time.sleep(2 ** attempt)
    raise RuntimeError(f"Cannot pin {dest.name}: {last_err}")


fetch_pinned([HERBENCH_BASE_URL, JSDELIVR_BASE], EVAL_DIR / "base_vlm.py", HERBENCH_BASE_SHA256)
fetch_pinned([HERBENCH_ACC_URL, JSDELIVR_ACC], EVAL_DIR / "calculate_accuracy.py", HERBENCH_ACC_SHA256)
print({"PERSIST": str(PERSIST), "OUT_ROOT": str(OUT_ROOT), "EVAL_DIR": str(EVAL_DIR)})


## 4. Clone source and verify


In [ ]:
# Fetch FrameWeave source. Run All clones the repo; do not upload .py files.
from __future__ import annotations

import subprocess
from pathlib import Path

REPO_URL = "https://github.com/nqMinh-GR/FrameWeave.git"
REPO_DIR = Path("/content/FrameWeave")
ROOT = Path("/content")
assert ROOT.exists(), "This notebook targets Google Colab (/content)."

if not (REPO_DIR / ".git").exists():
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)])
else:
    subprocess.check_call(["git", "-C", str(REPO_DIR), "pull", "--ff-only"])


def link(src: Path, dest: Path) -> None:
    src = src.resolve()
    if dest.is_symlink():
        if dest.resolve() != src:
            dest.unlink()
            dest.symlink_to(src)
        return
    if dest.exists():
        raise RuntimeError(
            f"{dest} already exists and is not the repo link. Restart the runtime, then Run All."
        )
    dest.symlink_to(src)


link(REPO_DIR / "ovis" / "reljoin", ROOT / "reljoin")
link(REPO_DIR / "ovis_ablation" / "ablation", ROOT / "ablation")
link(REPO_DIR / "ovis_ablation" / "configs", ROOT / "configs")
link(REPO_DIR / "ovis_ablation" / "run_ablation.py", ROOT / "run_ablation.py")
print("FrameWeave source linked from", REPO_DIR / "ovis_ablation", "and", REPO_DIR / "ovis" / "reljoin")


In [ ]:
# 4. Verify ablation sources (eval needs the runner and the V6 package)
from __future__ import annotations
from pathlib import Path

ROOT = Path("/content")
required = [
    ROOT / "run_ablation.py",
    ROOT / "configs" / "ovis_multiseed.yaml",
    ROOT / "ablation" / "eval_herbench.py",
    ROOT / "ablation" / "aggregate.py",
    ROOT / "reljoin" / "reljoin_modules.py",
    ROOT / "reljoin" / "herbench_oracle.py",
    ROOT / "reljoin" / "ovis_generate.py",
    ROOT / "herbench_eval_pins" / "base_vlm.py",
    ROOT / "herbench_eval_pins" / "calculate_accuracy.py",
]
missing = [str(path) for path in required if not path.is_file()]
assert not missing, "Missing files — clone or symlink failed:\n" + "\n".join(missing)
print("sources OK", len(required))


## 5. Gắn checkpoint `best/adapter.pt`

Mỗi seed cần `global_only`, `local_only` và `hybrid`. Thiếu file thì dừng và in đúng path.

Ưu tiên `checkpoints/ovis_multiseed/` trong repo đã clone. Drive chỉ bù file còn thiếu.


In [ ]:
# Prefer the cloned checkpoint. Drive fills only files that are still missing.
from __future__ import annotations

import shutil
from pathlib import Path

ROOT = Path("/content")
REPO_DIR = Path("/content/FrameWeave")
PUBLISHED = REPO_DIR / "checkpoints" / "ovis_multiseed"
OUT_ROOT = ROOT / "outputs" / "frameweave_ovis_multiseed"
try:
    SEEDS
except NameError:
    SEEDS = [1, 100, 2027]
try:
    VARIANT_LIST
except NameError:
    VARIANT_LIST = ["global_only", "local_only", "hybrid"]
try:
    PERSIST
except NameError:
    PERSIST = Path("/content/drive/MyDrive/frameweave_ovis_multiseed_outputs")

def link_published(src: Path, dest: Path) -> None:
    if not src.is_file():
        return
    dest.parent.mkdir(parents=True, exist_ok=True)
    src = src.resolve()
    if dest.is_symlink():
        if dest.resolve() != src:
            dest.unlink()
            dest.symlink_to(src)
        return
    if dest.exists():
        return
    dest.symlink_to(src)


def copy_missing(remote: Path, local: Path) -> int:
    if not remote.is_dir() or remote.resolve() == local.resolve():
        return 0
    copied = 0
    for src in remote.rglob("*"):
        if not src.is_file():
            continue
        dest = local / src.relative_to(remote)
        if dest.exists() or dest.is_symlink():
            continue
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dest)
        copied += 1
    return copied


linked = 0
for seed in SEEDS:
    for name in VARIANT_LIST:
        for bucket in ("best", "last"):
            src = PUBLISHED / f"seed_{seed}" / name / bucket / "adapter.pt"
            dest = OUT_ROOT / f"seed_{seed}" / f"seed_{seed}" / name / bucket / "adapter.pt"
            link_published(src, dest)
            if dest.is_file():
                linked += 1
    copied = copy_missing(PERSIST / f"seed_{seed}", OUT_ROOT / f"seed_{seed}")
    print({"seed": seed, "drive_files_copied": copied})
print("published adapter links", linked)


In [ ]:
# 5. Require best adapters for every seed and variant. Do not train here.
from __future__ import annotations
from pathlib import Path

ROOT = Path("/content")
OUT_ROOT = ROOT / "outputs" / "frameweave_ovis_multiseed"
DRIVE = Path("/content/drive/MyDrive/frameweave_ovis_multiseed_outputs")
try:
    SEEDS
except NameError:
    SEEDS = [1, 100, 2027]
try:
    VARIANT_LIST
except NameError:
    VARIANT_LIST = ["global_only", "local_only", "hybrid"]

missing = []
for seed in SEEDS:
    for name in VARIANT_LIST:
        local = OUT_ROOT / f"seed_{seed}" / f"seed_{seed}" / name / "best" / "adapter.pt"
        remote = DRIVE / f"seed_{seed}" / f"seed_{seed}" / name / "best" / "adapter.pt"
        if not local.is_file():
            missing.append(f"local {local}\ndrive {remote}")
if missing:
    raise FileNotFoundError(
        "Thiếu best checkpoint. Train notebook phải copy seed_1, seed_100 và seed_2027 lên Drive trước.\n"
        + "\n".join(missing)
    )
print("checkpoints OK", [(seed, name) for seed in SEEDS for name in VARIANT_LIST])


## 6. Tải Ovis2.5-9B + HERBench lite videos (~35 GB)

PerceptionTest không tải — không train.


In [ ]:
# 6. Download Ovis2.5-9B + HERBench lite (eval only)
from __future__ import annotations

import os
import shutil
import subprocess
from pathlib import Path

ROOT = Path("/content")
PYTHON = ROOT / "ovis_env" / "bin" / "python"
assert PYTHON.exists(), "Chạy cell 2 trước"
os.environ["PATH"] = f"{ROOT / 'ovis_env' / 'bin'}:{os.environ.get('PATH', '')}"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

HERBENCH_ROOT = ROOT / "HERBench_official"
HERBENCH_VIDEOS = HERBENCH_ROOT / "videos"
MODEL_DIR = ROOT / "checkpoints" / "ovis2_5_9b"
HERBENCH_HF_ID = "DanBenAmi/HERBench"
HERBENCH_REVISION = "30b42e97180f2c7c0fdc3f2aed412ae04f0366ca"
MODEL_HF_ID = "ATH-MaaS/Ovis2.5-9B"
MODEL_REVISION = "5a1f9b995c98ae5b85a28ec89d88d74295be0ea7"

for path in [HERBENCH_ROOT, HERBENCH_VIDEOS, MODEL_DIR.parent]:
    path.mkdir(parents=True, exist_ok=True)

print("Downloading Ovis2.5-9B (~18 GB bf16)...")
subprocess.run([str(PYTHON), "-c", f"""
import os
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
from huggingface_hub import snapshot_download
snapshot_download(
    repo_id='{MODEL_HF_ID}',
    revision='{MODEL_REVISION}',
    local_dir=r'{MODEL_DIR}',
    ignore_patterns=['*.msgpack', '*.h5', 'flax_model*', 'tf_model*'],
)
print('model OK')
"""], check=True)
assert (MODEL_DIR / "config.json").exists(), f"Missing {MODEL_DIR / 'config.json'}"

print("Downloading HERBench lite (~35GB)...")
subprocess.run([str(PYTHON), "-c", f"""
import os
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
from huggingface_hub import snapshot_download
snapshot_download(
    repo_id='{HERBENCH_HF_ID}',
    repo_type='dataset',
    revision='{HERBENCH_REVISION}',
    local_dir=r'{HERBENCH_ROOT}',
    allow_patterns=[
        'data/herbench_lite_v2.parquet',
        'data/herbench_annotations_lite_v2.json',
        'data/*metadata.json',
        'videos/videos.tar.part.00',
        'videos/videos.tar.part.01',
        'videos/videos.tar.part.02',
        'videos/videos.tar.part.03',
        'videos/videos_lite_info.txt',
        'videos/videos.tar.checksums.txt',
    ],
)
print('herbench snapshot OK')
"""], check=True)

LITE_MP4_MIN = 68
parts = sorted(HERBENCH_VIDEOS.glob("videos.tar.part.*"))
print("parts:", [path.name for path in parts])
mp4s = list(HERBENCH_VIDEOS.rglob("*.mp4"))
assert len(parts) >= 4 or len(mp4s) >= 2, (
    f"Thiếu videos.tar.part.00–03 trong {HERBENCH_VIDEOS}. Có: {[path.name for path in parts]}"
)
if len(mp4s) < LITE_MP4_MIN:
    assert len(parts) >= 4, (
        f"Chỉ có {len(mp4s)} mp4 và thiếu tar parts — tải lại videos.tar.part.00–03."
    )
    cat = subprocess.Popen(["cat", *map(str, parts)], stdout=subprocess.PIPE)
    subprocess.run(["tar", "-xf", "-", "-C", str(HERBENCH_VIDEOS)], stdin=cat.stdout, check=True)
    cat.wait()
    print("Extracted archives")
else:
    print(f"Already have {len(mp4s)} mp4")

n_mp4 = len(list(HERBENCH_VIDEOS.rglob("*.mp4")))
if n_mp4 >= LITE_MP4_MIN:
    for part in parts:
        part.unlink(missing_ok=True)
    print("Removed archives after full extract")
else:
    print(f"KEEP tar parts — only {n_mp4} mp4 (need ~{LITE_MP4_MIN} for full eval)")

print({
    "videos_mp4": n_mp4,
    "model_config": (MODEL_DIR / "config.json").exists(),
    "free_GiB": round(shutil.disk_usage(ROOT).free / 2**30, 2),
})
assert (MODEL_DIR / "config.json").exists(), "Ovis config.json missing — tải lại cell 6."
assert n_mp4 >= 2, "Lite extract xong vẫn thiếu mp4"
if n_mp4 < LITE_MP4_MIN:
    print(f"WARN: {n_mp4} mp4 < {LITE_MP4_MIN}. Smoke có thể chạy; full eval sẽ thiếu video.")
print("Downloads OK")


## 7. Eval full lite_v2 từng seed

Không train. Prediction nằm ở `seed_<seed>/eval/seed_<seed>/predictions.jsonl`. Xong seed nào thì copy thư mục seed lên Drive để lần chạy sau nối tiếp câu chưa xong.


In [ ]:
# 7. HERBench oracle eval for seeds 1, 100, 2027. No training.
from __future__ import annotations
import json
import os
import shutil
import subprocess
from pathlib import Path

ROOT = Path("/content")
PYTHON = ROOT / "ovis_env" / "bin" / "python"
OUT_ROOT = ROOT / "outputs" / "frameweave_ovis_multiseed"
CONFIG_DIR = ROOT / "configs" / "multiseed"
assert PYTHON.exists(), "Chạy cell cài ovis_env trước"
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ.setdefault("RELJOIN_ROOT", str(ROOT))
os.environ.setdefault("OVIS_ABLATION_ROOT", str(ROOT))

try:
    SEEDS
except NameError:
    SEEDS = [1, 100, 2027]
try:
    VARIANT
except NameError:
    VARIANT = "global_only,local_only,hybrid"
try:
    PERSIST
except NameError:
    PERSIST = Path("/content/drive/MyDrive/frameweave_ovis_multiseed_outputs")
    if not PERSIST.parent.exists():
        PERSIST = OUT_ROOT / "persist_local"

def _yaml_scalar(value):
    if value is None:
        return "null"
    if isinstance(value, bool):
        return "true" if value else "false"
    if isinstance(value, float):
        if value.is_integer():
            return f"{value:.1f}"
        return format(value, ".12g")
    if isinstance(value, int):
        return str(value)
    if isinstance(value, str):
        if value == "":
            return '""'
        if any(ch in value for ch in ":#[]{}&*!|>%@`'\""):
            return json.dumps(value, ensure_ascii=False)
        return value
    raise TypeError(type(value))


def dump_multiseed_yaml(raw: dict) -> str:
    lines = [
        "# Generated by the V6 multi-seed notebook.",
        "# Training seeds are 1, 100, 2027. The PerceptionTest split is the canonical seed-42 split.",
    ]
    for key, value in raw.items():
        if isinstance(value, dict):
            lines.append(f"{key}:")
            for sub, subval in value.items():
                lines.append(f"  {sub}: {_yaml_scalar(subval)}")
        elif isinstance(value, list):
            inner = ", ".join(_yaml_scalar(item) for item in value)
            lines.append(f"{key}: [{inner}]")
        else:
            lines.append(f"{key}: {_yaml_scalar(value)}")
    return "\n".join(lines) + "\n"


def write_multiseed_yaml(path: Path, *, seed: int, output_root: Path, paths: dict[str, str]) -> None:
    import sys
    sys.path.insert(0, str(ROOT))
    from ablation.config import load_yaml

    raw = load_yaml(ROOT / "configs" / "ovis_multiseed.yaml")
    raw["seed"] = int(seed)
    raw["variants"] = ["global_only", "local_only", "hybrid"]
    raw["output_root"] = str(output_root)
    merged = dict(raw.get("paths") or {})
    merged.update(paths)
    raw["paths"] = merged
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(dump_multiseed_yaml(raw), encoding="utf-8")
    print("Wrote", path, {"seed": seed, "output_root": str(output_root)})

def eval_paths() -> dict[str, str]:
    return {
        "model_dir": "/content/checkpoints/ovis2_5_9b",
        "herbench_root": "/content/HERBench_official",
        "herbench_videos": "/content/HERBench_official/videos",
        "herbench_base_vlm": "/content/herbench_eval_pins/base_vlm.py",
        "herbench_accuracy": "/content/herbench_eval_pins/calculate_accuracy.py",
    }

def publish(seed_out: Path, seed: int) -> None:
    dest = PERSIST / f"seed_{seed}"
    if not PERSIST.parent.exists() and PERSIST.name != "persist_local":
        print("Drive not mounted; skip publish", seed)
        return
    if dest.resolve() == seed_out.resolve():
        return
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(seed_out, dest, dirs_exist_ok=True)
    print("Copied", seed_out, "→", dest)

for seed in SEEDS:
    seed_out = OUT_ROOT / f"seed_{seed}"
    cfg_path = CONFIG_DIR / f"eval_seed_{seed}.yaml"
    write_multiseed_yaml(cfg_path, seed=seed, output_root=seed_out, paths=eval_paths())
    cmd = [
        str(PYTHON), "run_ablation.py", "eval",
        "--config", str(cfg_path),
        "--variant", VARIANT,
        "--seed", str(seed),
    ]
    print("$", " ".join(cmd), flush=True)
    subprocess.run(cmd, cwd=str(ROOT), check=True)
    prediction = seed_out / "eval" / f"seed_{seed}" / "predictions.jsonl"
    metrics = seed_out / "eval" / f"seed_{seed}" / "metrics.json"
    assert prediction.is_file(), prediction
    payload = json.loads(metrics.read_text(encoding="utf-8"))
    print({"seed": seed, "n_predictions": payload.get("n_predictions"), "complete": payload.get("complete")})
    publish(seed_out, seed)
print("Eval loop finished")


## 8. Bảng từng seed và mean ± std

Std là độ lệch chuẩn mẫu (chia cho $n-1$) của ba seed. `Complete=no` chưa phải lite_v2 đầy đủ. Mean chỉ là kết quả ba seed khi cả ba dòng đều `Complete=yes`.


In [ ]:
# 8. Per-seed table plus mean and sample std. Does not call the six-variant aggregate.
from __future__ import annotations
import os
import subprocess
from pathlib import Path

ROOT = Path("/content")
PYTHON = ROOT / "ovis_env" / "bin" / "python"
assert PYTHON.exists(), "Chạy cell cài ovis_env trước"
os.environ.setdefault("RELJOIN_ROOT", str(ROOT))
os.environ.setdefault("OVIS_ABLATION_ROOT", str(ROOT))
script = ROOT / "outputs" / "frameweave_ovis_multiseed" / "aggregate_seeds.py"
script.parent.mkdir(parents=True, exist_ok=True)
script.write_text(
    r"""
import csv
import json
import math
import sys
from pathlib import Path

sys.path.insert(0, "/content")
from ablation.aggregate import summarize_predictions
from ablation.common import read_json, read_jsonl
from ablation.config import TASKS

SEEDS = [1, 100, 2027]
VARIANTS = ["global_only", "local_only", "hybrid"]
ROOT = Path("/content/outputs/frameweave_ovis_multiseed")
EXPECTED = 1971
FAMILIES = ["TR&C", "R&T", "GC&V", "ME&N"]
METRIC_KEYS = [
    ("overall_avg", "Overall Avg."),
    ("qa_accuracy", "QA Acc."),
    ("tso", "TSO"),
    ("non_tso_avg", "Non-TSO Avg."),
]


def pct(value):
    if value is None:
        return ""
    return f"{100.0 * float(value):.2f}"


def metric_values(summary):
    values = {key: summary.get(key) for key, _label in METRIC_KEYS}
    families = summary.get("families") or {}
    for name in FAMILIES:
        values[name] = families.get(name)
    return values


def mean_std(nums):
    if not nums:
        return None, None
    mean = sum(nums) / len(nums)
    if len(nums) < 2:
        return mean, None
    var = sum((value - mean) ** 2 for value in nums) / (len(nums) - 1)
    return mean, math.sqrt(var)


summaries = []
for seed in SEEDS:
    out = ROOT / f"seed_{seed}"
    pred = out / "eval" / f"seed_{seed}" / "predictions.jsonl"
    if not pred.is_file():
        raise FileNotFoundError(f"Missing predictions: {pred}")
    rows = [row for row in read_jsonl(pred) if int(row.get("seed", -1)) == int(seed)]
    for name in VARIANTS:
        selected_path = out / f"seed_{seed}" / name / "selected.json"
        budget_path = out / f"seed_{seed}" / name / "parameter_manifest.json"
        selected = read_json(selected_path) if selected_path.is_file() else {}
        budget = read_json(budget_path) if budget_path.is_file() else {}
        checkpoint_ids = sorted({
            str(row.get("checkpoint_id") or "")
            for row in rows
            if row.get("variant") == name and row.get("checkpoint_id")
        })
        summary = summarize_predictions(
            rows,
            variant=name,
            seed=seed,
            expected_questions=EXPECTED,
            checkpoint_id=checkpoint_ids[-1] if checkpoint_ids else "",
            selected_epoch=selected.get("epoch"),
            trainable_params=budget.get("trainable"),
        )
        summaries.append(summary)

columns = [
    "Variant",
    "Seed",
    "Trainable Params",
    "Selected Epoch",
    "Overall Avg.",
    "QA Acc.",
    "TSO",
    "Non-TSO Avg.",
    "TR&C",
    "R&T",
    "GC&V",
    "ME&N",
    "Coverage",
    "Questions",
    "Videos",
    "Complete",
]


def row_from_summary(summary):
    values = metric_values(summary)
    return {
        "Variant": summary["variant"],
        "Seed": str(summary["seed"]),
        "Trainable Params": "" if summary["trainable_params"] is None else str(summary["trainable_params"]),
        "Selected Epoch": "" if summary["selected_epoch"] is None else str(summary["selected_epoch"]),
        "Overall Avg.": pct(values["overall_avg"]),
        "QA Acc.": pct(values["qa_accuracy"]),
        "TSO": pct(values["tso"]),
        "Non-TSO Avg.": pct(values["non_tso_avg"]),
        "TR&C": pct(values["TR&C"]),
        "R&T": pct(values["R&T"]),
        "GC&V": pct(values["GC&V"]),
        "ME&N": pct(values["ME&N"]),
        "Coverage": pct(summary["coverage"]),
        "Questions": str(summary["n_ok"]),
        "Videos": str(summary["n_videos"]),
        "Complete": "yes" if summary["complete"] else "no",
    }


table_rows = [row_from_summary(summary) for summary in summaries]
aggregate_raw = []
for name in VARIANTS:
    grouped = [summary for summary in summaries if summary["variant"] == name]
    complete = len(grouped) == len(SEEDS) and all(summary["complete"] for summary in grouped)
    means = {"variant": name, "seed": "mean", "complete": complete}
    stds = {"variant": name, "seed": "std", "complete": complete}
    keys = [key for key, _label in METRIC_KEYS] + FAMILIES
    for key in keys:
        nums = []
        for summary in grouped:
            value = metric_values(summary)[key]
            if value is not None:
                nums.append(float(value))
        mean, std = mean_std(nums)
        means[key] = mean
        stds[key] = std
        if len(nums) != len(SEEDS):
            means["complete"] = False
            stds["complete"] = False
    aggregate_raw.append(means)
    aggregate_raw.append(stds)
    for kind, payload in (("mean", means), ("std", stds)):
        table_rows.append({
            "Variant": name,
            "Seed": kind,
            "Trainable Params": "",
            "Selected Epoch": "",
            "Overall Avg.": pct(payload["overall_avg"]),
            "QA Acc.": pct(payload["qa_accuracy"]),
            "TSO": pct(payload["tso"]),
            "Non-TSO Avg.": pct(payload["non_tso_avg"]),
            "TR&C": pct(payload["TR&C"]),
            "R&T": pct(payload["R&T"]),
            "GC&V": pct(payload["GC&V"]),
            "ME&N": pct(payload["ME&N"]),
            "Coverage": "",
            "Questions": "",
            "Videos": "",
            "Complete": "yes" if payload["complete"] else "no",
        })

out_dir = ROOT / "aggregate"
out_dir.mkdir(parents=True, exist_ok=True)
csv_path = out_dir / "results.csv"
with csv_path.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=columns + list(TASKS))
    writer.writeheader()
    for summary, rendered in zip(summaries, table_rows[: len(summaries)]):
        extra = {}
        for task, stats in summary["per_task"].items():
            extra[task] = "" if stats is None else pct(stats["accuracy"])
        writer.writerow({**rendered, **extra})
    empty_tasks = {task: "" for task in TASKS}
    for rendered in table_rows[len(summaries) :]:
        writer.writerow({**rendered, **empty_tasks})

header = "| " + " | ".join(columns) + " |"
rule = "| " + " | ".join("---" for _ in columns) + " |"
lines = [
    header,
    rule,
]
for rendered in table_rows:
    lines.append("| " + " | ".join(rendered[column] for column in columns) + " |")
lines.extend([
    "",
    "Seeds are 1, 100, and 2027. Std is the sample standard deviation (ddof=1) across those seeds.",
    "Non-TSO Avg. is the unweighted mean of the 11 non-TSO task accuracies.",
    "It is a diagnostic, not the micro accuracy after dropping TSO.",
    "Overall Avg. is the unweighted mean of all 12 tasks. QA Acc. is micro accuracy.",
    "Invalid generations stay in the QA denominator.",
    "A per-seed row with Complete=no is not a full lite_v2 result.",
    "A mean/std row has Complete=yes only when all three seeds are complete for that variant.",
    "Paper reference numbers are not tuning targets: Vanilla Overall 61.19 / QA 56.82; Hybrid Overall 66.54 / QA 60.98.",
    "",
    "Per-task accuracy:",
    "",
    "| Variant | Seed | " + " | ".join(TASKS) + " |",
    "| --- | --- | " + " | ".join("---" for _ in TASKS) + " |",
])
for summary in summaries:
    tasks = []
    for task in TASKS:
        stats = summary["per_task"][task]
        tasks.append("" if stats is None else pct(stats["accuracy"]))
    lines.append("| " + summary["variant"] + " | " + str(summary["seed"]) + " | " + " | ".join(tasks) + " |")
md_path = out_dir / "results.md"
md_path.write_text(chr(10).join(lines) + chr(10), encoding="utf-8")
(out_dir / "summaries.json").write_text(
    json.dumps({"per_seed": summaries, "mean_std": aggregate_raw}, indent=2),
    encoding="utf-8",
)
print(md_path.read_text(encoding="utf-8"))
print("Wrote", csv_path)
""",
    encoding="utf-8",
)
print("$", PYTHON, script)
subprocess.run([str(PYTHON), str(script)], cwd=str(ROOT), check=True)


## 9. Copy artifact sang Drive


In [ ]:
# 9. Persist eval predictions and the multiseed aggregate to Drive.
from __future__ import annotations
import shutil
from datetime import datetime, timezone
from pathlib import Path

ROOT = Path("/content")
OUT_ROOT = ROOT / "outputs" / "frameweave_ovis_multiseed"
persist = Path("/content/drive/MyDrive/frameweave_ovis_multiseed_outputs")
if not persist.parent.exists():
    persist = OUT_ROOT / "persist_local"
persist.mkdir(parents=True, exist_ok=True)
if persist.resolve() != OUT_ROOT.resolve():
    aggregate = OUT_ROOT / "aggregate"
    if aggregate.exists():
        shutil.copytree(aggregate, persist / "aggregate", dirs_exist_ok=True)
        print("Copied", aggregate, "→", persist / "aggregate")
    for seed in (1, 100, 2027):
        src = OUT_ROOT / f"seed_{seed}"
        if not src.exists():
            print("skip missing", src)
            continue
        dest = persist / f"seed_{seed}"
        shutil.copytree(src, dest, dirs_exist_ok=True)
        print("Copied", src, "→", dest)

stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
zip_path = ROOT / f"frameweave_ovis_multiseed_eval_{stamp}"
if OUT_ROOT.exists():
    shutil.make_archive(str(zip_path), "zip", root_dir=OUT_ROOT)
    print("Zip:", f"{zip_path}.zip")
print("Persist:", persist)
